# RAG mode: document QA with `RAGSystem`

`MemorySystem` is built for *memory* — dated facts distilled from an ongoing stream. When you have a static document corpus and want classic retrieval-augmented QA, `RAGSystem` skips distillation and indexes passages directly, with the same bucket-gated retrieval underneath (plus optional multi-hop expansion and query decomposition).

This notebook indexes the bundled contract demo and compares plain dense retrieval against the bucket-gated `coremem` method.

In [ ]:
from pathlib import Path

# Build a passage corpus: one passage per contract section.
corpus = []
for f in sorted(Path("../demos/contract-qa").glob("*.md")):
    text = f.read_text()
    title = text.splitlines()[0].lstrip("# ")
    for section in text.split("\n## ")[1:]:
        heading, _, body = section.partition("\n")
        corpus.append({"title": f"{title} — {heading}", "text": body.strip()})

print(f"{len(corpus)} passages")
for c in corpus[:5]:
    print(" -", c["title"])

In [ ]:
from membukkit import RAGSystem, RAGConfig

rag = RAGSystem.from_pretrained(
    rag_cfg=RAGConfig(method="dense", top_k=4),
    llm="openai:gpt-4o-mini",
)
rag.index(corpus)

result = rag.answer("How quickly must CloudVault notify Meridian of a data breach?")
print("ANSWER:", result.answer)
print("\nretrieved passages:")
for title in result.trace.ranked_titles:
    print(" -", title)

Note the answer above: the amendment (24 hours) supersedes the original MSA clause (48 hours) — the reader has to reconcile both retrieved passages.

## Bucket-gated retrieval (`method="coremem"`)

On large corpora, dense retrieval embeds the query against *every* passage. The `coremem` method clusters passages into buckets and routes, scanning only a budgeted fraction — the same efficiency lever as the memory pipeline. It can also do multi-hop expansion (`hops=2`) for questions whose answer spans linked passages.

In [ ]:
rag2 = RAGSystem.from_pretrained(
    rag_cfg=RAGConfig(method="coremem", budget=0.6, bucket_k=6, top_k=4),
    llm="openai:gpt-4o-mini",
)
rag2.index(corpus)

for q in [
    "What service credits apply if uptime drops to 99.5%?",
    "Can Meridian terminate early without breach, and what does it cost?",
]:
    r = rag2.answer(q)
    print("Q:", q)
    print("A:", r.answer)
    print("   passages:", "; ".join(t[:50] for t in r.trace.ranked_titles[:3]), "\n")

## When to use which

| | `MemorySystem` | `RAGSystem` |
|---|---|---|
| Data | ongoing stream (chats, tickets, logs) | static document corpus |
| Unit | distilled dated facts + raw turns | passages as-is |
| Strength | temporal reasoning, knowledge updates | multi-hop document QA |
| Persistence | local stores / Turbopuffer | in-process index |

Both share the encoder, reranker, and bucket-routing machinery. Full config surface: `RAGConfig` (decomposition, hops, entity expansion) and `RetrievalConfig` (lanes, budgets, buckets).